# 🤖 Machine Learning: A–Z Concepts for Data Scientists
## A Comprehensive Illustrated Reference Notebook

> **How to use this notebook:** Every section combines a plain-English explanation, a real-world analogy, working code on demo data, and a visualisation.  
> Run cells top-to-bottom. Each section is self-contained.

---

### 📚 Table of Contents

| Letter | Concept |
|--------|---------|
| A | Accuracy, Activation Functions, Anomaly Detection |
| B | Bias–Variance Trade-off, Bagging, Boosting |
| C | Cross-Validation, Confusion Matrix, Clustering |
| D | Decision Trees, Dimensionality Reduction |
| E | Ensemble Methods, Evaluation Metrics |
| F | Feature Engineering, Feature Scaling (StandardScaler & MinMaxScaler), Feature Selection |
| G | Gradient Descent |
| H | Hyperparameter Tuning |
| I | Imbalanced Data, Imputation |
| K | K-Nearest Neighbours, K-Means Clustering |
| L | Label Encoding, Linear Regression, Logistic Regression, Loss Functions |
| M | Model Evaluation, Model Persistence |
| N | Normalisation |
| O | Overfitting & Underfitting, One-Hot Encoding |
| P | Pipelines, Precision & Recall |
| R | Regularisation (L1 / L2), Random Forests, ROC-AUC |
| S | Support Vector Machines, Stratified Sampling |
| T | Train / Validation / Test Split, Transfer Learning |
| V | Variance, Validation Curves |
| W | Weights & Initialisation |

---


## ⚙️ Global Setup — Run First
Imports and demo datasets used throughout the notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import (make_classification, make_regression,
                               make_blobs, load_iris, load_breast_cancer)
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      validation_curve, GridSearchCV,
                                      StratifiedKFold)
from sklearn.preprocessing import (StandardScaler, MinMaxScaler,
                                    LabelEncoder, OneHotEncoder,
                                    PolynomialFeatures)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import (LinearRegression, LogisticRegression,
                                   Ridge, Lasso)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               BaggingClassifier, AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, roc_curve, auc,
                              mean_squared_error, r2_score,
                              precision_score, recall_score, f1_score)
from sklearn.impute import SimpleImputer
import joblib, os

# ── Consistent style ──────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#F5F5F5',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
PALETTE = ['#2E86AB','#E84855','#3BB273','#F4A261','#A23B72','#F18F01']

# ── Reproducibility ───────────────────────────────────────
np.random.seed(42)

print("✅ All imports successful. Ready to learn!")


---
## 🔤 A

### A1 — Accuracy
**Definition:** Accuracy is the fraction of predictions that are correct.

$$\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}} = \frac{TP + TN}{TP + TN + FP + FN}$$

**⚠️ Caution:** Accuracy is misleading on imbalanced datasets. If 95 % of emails are not spam, a model that always predicts "not spam" achieves 95 % accuracy — yet it's useless.


In [ ]:
# Demo: Accuracy on balanced vs imbalanced data
from sklearn.datasets import make_classification

X_bal, y_bal = make_classification(n_samples=1000, weights=[0.5, 0.5], random_state=42)
X_imb, y_imb = make_classification(n_samples=1000, weights=[0.95, 0.05], random_state=42)

for name, (X, y) in [("Balanced (50/50)", (X_bal, y_bal)),
                      ("Imbalanced (95/5)", (X_imb, y_imb))]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LogisticRegression(max_iter=200)
    model.fit(X_tr, y_tr)
    naive_acc  = max(np.bincount(y_te)) / len(y_te)
    model_acc  = accuracy_score(y_te, model.predict(X_te))
    print(f"{name}:")
    print(f"  Naive (predict majority) accuracy : {naive_acc:.2%}")
    print(f"  Logistic Regression accuracy       : {model_acc:.2%}")
    print()


### A2 — Activation Functions
**Definition:** An activation function introduces non-linearity into a neural network layer, allowing it to learn complex patterns.

| Function | Formula | When to use |
|----------|---------|-------------|
| Sigmoid  | 1 / (1 + e⁻ˣ) | Binary output (0–1) |
| ReLU     | max(0, x) | Hidden layers (default) |
| Tanh     | (eˣ − e⁻ˣ)/(eˣ + e⁻ˣ) | Zero-centred alternative to sigmoid |
| Softmax  | eˣᵢ / Σeˣʲ | Multi-class output layer |


In [ ]:
x = np.linspace(-5, 5, 300)

sigmoid = 1 / (1 + np.exp(-x))
relu    = np.maximum(0, x)
tanh    = np.tanh(x)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Activation Functions", fontsize=14, fontweight='bold')

for ax, (name, y, col) in zip(axes, [
        ("Sigmoid", sigmoid, PALETTE[0]),
        ("ReLU",    relu,    PALETTE[1]),
        ("Tanh",    tanh,    PALETTE[2])]):
    ax.plot(x, y, color=col, linewidth=2.5)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel("Input (x)")
    ax.set_ylabel("Output")

plt.tight_layout()
plt.savefig("activation_functions.png", dpi=120, bbox_inches='tight')
plt.show()
print("💡 ReLU is the most widely-used in deep networks because it avoids the vanishing gradient problem.")


---
## 🔤 B

### B1 — Bias–Variance Trade-off
**Bias** = Error from wrong assumptions (too simple model → underfits).  
**Variance** = Error from sensitivity to small fluctuations in training data (too complex model → overfits).  
**Goal:** Find the sweet spot — low bias AND low variance.

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$


In [ ]:
# Visualise bias-variance with polynomial regression
from sklearn.pipeline import make_pipeline

np.random.seed(42)
X_true = np.linspace(0, 1, 100)
y_true = np.sin(2 * np.pi * X_true)
X_train = np.random.choice(X_true, 20, replace=False)
y_train = np.sin(2 * np.pi * X_train) + np.random.normal(0, 0.2, 20)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Bias–Variance Trade-off", fontsize=14, fontweight='bold')

for ax, (deg, label, col) in zip(axes, [
        (1,  "Degree 1 — High Bias\n(Underfitting)", PALETTE[0]),
        (4,  "Degree 4 — Good Fit\n(Just Right)",    PALETTE[2]),
        (15, "Degree 15 — High Variance\n(Overfitting)", PALETTE[1])]):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_train.reshape(-1,1), y_train)
    y_pred = model.predict(X_true.reshape(-1,1))
    ax.scatter(X_train, y_train, color='gray', alpha=0.7, zorder=5, label='Training data', s=30)
    ax.plot(X_true, y_true, 'k--', linewidth=1.5, label='True function')
    ax.plot(X_true, y_pred, color=col, linewidth=2.5, label=f'Degree {deg}')
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylim(-2, 2)

plt.tight_layout()
plt.savefig("bias_variance.png", dpi=120, bbox_inches='tight')
plt.show()


### B2 — Bagging vs Boosting

| | Bagging | Boosting |
|--|---------|---------|
| **Idea** | Train many models in **parallel** on random subsets | Train models **sequentially**, each fixing the previous one's errors |
| **Goal** | Reduce **variance** | Reduce **bias** |
| **Example** | Random Forest | AdaBoost, Gradient Boosting, XGBoost |
| **Overfitting risk** | Low | Medium–High (with too many rounds) |


In [ ]:
X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

models = {
    "Single Decision Tree": DecisionTreeClassifier(random_state=42),
    "Bagging (Random Forest)":  RandomForestClassifier(n_estimators=100, random_state=42),
    "Boosting (AdaBoost)":       AdaBoostClassifier(n_estimators=100, random_state=42, algorithm='SAMME'),
    "Boosting (Gradient Boost)": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print(f"{'Model':<30} {'Train Acc':>10} {'Test Acc':>10}")
print("-" * 52)
for name, m in models.items():
    m.fit(X_tr, y_tr)
    print(f"{name:<30} {accuracy_score(y_tr, m.predict(X_tr)):>10.2%} {accuracy_score(y_te, m.predict(X_te)):>10.2%}")


---
## 🔤 C

### C1 — Cross-Validation (k-Fold)
**Problem:** A single train/test split is unreliable — you might get lucky or unlucky with how data is split.  
**Solution:** k-Fold CV splits the data into k folds, trains on k-1 and evaluates on 1, rotating until every fold has been the test set. Reports mean ± std of the score.


In [ ]:
X, y = load_iris(return_X_y=True)

models_cv = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=50, random_state=42),
    "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5),
}

print(f"{'Model':<25} {'5-Fold CV Accuracy':>22}")
print("-" * 50)
cv_results = {}
for name, m in models_cv.items():
    scores = cross_val_score(m, X, y, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:<25} {scores.mean():.4f}  ±  {scores.std():.4f}")

# Visualise
fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
           boxprops=dict(facecolor=PALETTE[0], alpha=0.6))
ax.set_title("5-Fold Cross-Validation Accuracy Distribution", fontsize=13, fontweight='bold')
ax.set_ylabel("Accuracy")
ax.set_ylim(0.8, 1.02)
plt.tight_layout()
plt.savefig("cross_validation.png", dpi=120, bbox_inches='tight')
plt.show()


### C2 — Confusion Matrix
A **confusion matrix** shows the breakdown of correct and incorrect predictions by class.

|  | Predicted Positive | Predicted Negative |
|--|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

- **Precision** = TP / (TP + FP) — "Of all things I called positive, how many were right?"
- **Recall** = TP / (TP + FN) — "Of all actual positives, how many did I find?"
- **F1** = 2 × (Precision × Recall) / (Precision + Recall) — Harmonic mean


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

cm = confusion_matrix(y_te, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Malignant','Benign'], yticklabels=['Malignant','Benign'])
axes[0].set_title("Confusion Matrix", fontweight='bold')
axes[0].set_ylabel("Actual"); axes[0].set_xlabel("Predicted")

# Per-class metrics
report = classification_report(y_te, y_pred, target_names=['Malignant','Benign'], output_dict=True)
metrics_df = pd.DataFrame(report).T.iloc[:2][['precision','recall','f1-score']]
metrics_df.plot(kind='bar', ax=axes[1], color=PALETTE[:3], edgecolor='white', width=0.6)
axes[1].set_title("Precision / Recall / F1 by Class", fontweight='bold')
axes[1].set_ylim(0, 1.1); axes[1].legend(); axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120, bbox_inches='tight')
plt.show()
print(classification_report(y_te, y_pred, target_names=['Malignant','Benign']))


---
## 🔤 D

### D1 — Decision Trees
A Decision Tree splits data by asking yes/no questions on features, forming a tree of rules. Leaf nodes hold the final prediction.

**Key hyperparameters:**
- `max_depth` — limits tree depth to prevent overfitting
- `min_samples_split` — minimum samples required to make a split
- `criterion` — how to measure split quality (`gini`, `entropy`)


In [ ]:
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

# Train a shallow tree for visualisation
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(dt, feature_names=load_iris().feature_names,
          class_names=load_iris().target_names,
          filled=True, rounded=True, ax=ax, fontsize=9)
ax.set_title("Decision Tree (max_depth=3) — Iris Dataset", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("decision_tree.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"Test Accuracy: {accuracy_score(y_te, dt.predict(X_te)):.2%}")


### D2 — Dimensionality Reduction (PCA)
**Problem:** High-dimensional data is hard to visualise, slow to train on, and often contains redundant features.  
**PCA** (Principal Component Analysis) projects data onto a lower-dimensional space that preserves maximum variance.

**Analogy:** PCA is like creating a shadow of a 3D object. The shadow (2D) loses some detail but captures the main shape.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
pca.fit(X_scaled)
explained = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Explained variance
axes[0].plot(range(1, len(explained)+1), explained, marker='o', color=PALETTE[0], linewidth=2)
axes[0].axhline(0.95, color=PALETTE[1], linestyle='--', label='95% threshold')
axes[0].set_xlabel("Number of Components")
axes[0].set_ylabel("Cumulative Explained Variance")
axes[0].set_title("PCA — Explained Variance", fontweight='bold')
axes[0].legend()

# 2D projection
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_scaled)
for cls, name, col in zip([0,1], ['Malignant','Benign'], [PALETTE[1], PALETTE[0]]):
    mask = y == cls
    axes[1].scatter(X_2d[mask,0], X_2d[mask,1], c=col, alpha=0.5, label=name, s=20)
axes[1].set_title("PCA 2D Projection — Breast Cancer", fontweight='bold')
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].legend()

plt.tight_layout()
plt.savefig("pca.png", dpi=120, bbox_inches='tight')
plt.show()
n95 = np.argmax(explained >= 0.95) + 1
print(f"Original dimensions: {X.shape[1]}  |  Components needed for 95% variance: {n95}")


---
## 🔤 F

### F1 — Feature Scaling: StandardScaler vs MinMaxScaler

Feature scaling transforms numerical features to a common range or distribution. It is **essential** for algorithms that are sensitive to the magnitude of features, such as:
- K-Nearest Neighbours (distance-based)
- Support Vector Machines
- Logistic / Linear Regression with regularisation
- Neural Networks
- Gradient Descent in general

**Why does it matter?**  
If feature A ranges from 0–1 and feature B ranges from 0–1,000,000, most algorithms will incorrectly treat B as far more important.

---

#### StandardScaler (Z-score normalisation)
$$z = \frac{x - \mu}{\sigma}$$
- Centers data at 0 with unit variance
- Does **not** bound values to a fixed range
- **Best for:** Algorithms assuming normally distributed data; when outliers are present

#### MinMaxScaler
$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
- Scales to a fixed range, typically [0, 1]
- Sensitive to outliers (one extreme value compresses all others)
- **Best for:** Neural networks, image pixel values, bounded ranges


In [ ]:
# Create demo dataset with very different feature scales
np.random.seed(42)
n = 200
salary   = np.random.normal(60000, 15000, n)   # 30k – 100k
age      = np.random.normal(35, 8, n)           # 15 – 65
score    = np.random.normal(72, 12, n)           # 40 – 100
df_raw = pd.DataFrame({'Salary ($)': salary, 'Age (yrs)': age, 'Score (pts)': score})

ss  = StandardScaler()
mms = MinMaxScaler()
df_std = pd.DataFrame(ss.fit_transform(df_raw),  columns=df_raw.columns)
df_mms = pd.DataFrame(mms.fit_transform(df_raw), columns=df_raw.columns)

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle("Feature Scaling: Raw vs StandardScaler vs MinMaxScaler", fontsize=14, fontweight='bold')

titles = ["Raw (unscaled)", "StandardScaler (Z-score)", "MinMaxScaler [0, 1]"]
datasets = [df_raw, df_std, df_mms]

for col_idx, (title, dataset) in enumerate(zip(titles, datasets)):
    for row_idx, feat in enumerate(df_raw.columns):
        ax = axes[row_idx][col_idx]
        ax.hist(dataset[feat], bins=30, color=PALETTE[col_idx], edgecolor='white', alpha=0.85)
        if row_idx == 0:
            ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_ylabel(feat, fontsize=9)
        mean_v = dataset[feat].mean()
        std_v  = dataset[feat].std()
        ax.axvline(mean_v, color='red', linewidth=1.5, linestyle='--', label=f'μ={mean_v:.2f}')
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig("feature_scaling.png", dpi=120, bbox_inches='tight')
plt.show()

print("Summary Statistics After Scaling:")
print(f"\n{'Metric':<12} {'Salary':>12} {'Age':>12} {'Score':>12}")
print("-" * 50)
for label, dataset in [("Raw", df_raw), ("StandardScaler", df_std), ("MinMaxScaler", df_mms)]:
    print(f"\n[{label}]")
    print(f"  Mean:  {dataset.mean().values[0]:>12.4f} {dataset.mean().values[1]:>12.4f} {dataset.mean().values[2]:>12.4f}")
    print(f"  Std:   {dataset.std().values[0]:>12.4f} {dataset.std().values[1]:>12.4f} {dataset.std().values[2]:>12.4f}")
    print(f"  Min:   {dataset.min().values[0]:>12.4f} {dataset.min().values[1]:>12.4f} {dataset.min().values[2]:>12.4f}")
    print(f"  Max:   {dataset.max().values[0]:>12.4f} {dataset.max().values[1]:>12.4f} {dataset.max().values[2]:>12.4f}")


In [ ]:
# Impact of scaling on KNN classification accuracy
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

results = {}
for name, scaler in [("No Scaling", None),
                      ("StandardScaler", StandardScaler()),
                      ("MinMaxScaler",   MinMaxScaler())]:
    if scaler:
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)
    else:
        X_tr_s, X_te_s = X_tr, X_te
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_tr_s, y_tr)
    results[name] = accuracy_score(y_te, knn.predict(X_te_s))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(results.keys(), results.values(), color=PALETTE[:3], edgecolor='white', width=0.5)
ax.set_ylim(0.85, 1.0)
ax.set_title("KNN Accuracy — Impact of Feature Scaling", fontweight='bold')
ax.set_ylabel("Test Accuracy")
for bar, val in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.2%}", ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig("scaling_impact.png", dpi=120, bbox_inches='tight')
plt.show()
print("\n💡 Key takeaway: Scaling can dramatically improve distance-based models like KNN.")


### F2 — Label Encoding vs One-Hot Encoding

Machine learning models require **numerical** inputs. Categorical features must be converted first.

| Method | When to use | Caution |
|--------|-------------|---------|
| **Label Encoding** | Ordinal categories (Small < Medium < Large) | Implies order — don't use for nominal data with tree-based models only |
| **One-Hot Encoding** | Nominal categories (City: Lagos, London, NYC) | Creates many columns for high-cardinality features |
| **Ordinal Encoding** | Explicitly ordered categories | Must manually specify order |


In [ ]:
# Demo dataframe with categorical columns
df_cat = pd.DataFrame({
    'Name':       ['Alice','Bob','Charlie','Diana','Eve','Frank'],
    'City':       ['Lagos','London','NYC','Lagos','NYC','London'],
    'Size':       ['Small','Large','Medium','Small','Large','Medium'],
    'Purchased':  ['Yes','No','Yes','Yes','No','Yes']
})
print("Original DataFrame:")
print(df_cat)
print()

# ── Label Encoding (Ordinal: Small < Medium < Large) ─────
le = LabelEncoder()
df_cat['Size_LabelEncoded'] = le.fit_transform(df_cat['Size'])
df_cat['Purchased_Encoded'] = le.fit_transform(df_cat['Purchased'])

# ── One-Hot Encoding (Nominal: City) ─────────────────────
df_ohe = pd.get_dummies(df_cat, columns=['City'], prefix='City', dtype=int)

print("After Label Encoding (Size, Purchased):")
print(df_ohe[['Name','Size','Size_LabelEncoded','Purchased','Purchased_Encoded']].to_string(index=False))
print()
print("After One-Hot Encoding (City):")
print(df_ohe[['Name','City_Lagos','City_London','City_NYC']].to_string(index=False))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# Label encoding mapping
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
axes[0].bar(mapping.keys(), mapping.values(), color=PALETTE[:len(mapping)], edgecolor='white')
axes[0].set_title("Label Encoding — 'Purchased'", fontweight='bold')
axes[0].set_ylabel("Encoded Value")

# One-hot
city_cols = ['City_Lagos','City_London','City_NYC']
df_ohe[city_cols].sum().plot(kind='bar', ax=axes[1], color=PALETTE[:3], edgecolor='white')
axes[1].set_title("One-Hot Encoding — 'City' Counts", fontweight='bold')
axes[1].set_ylabel("Count"); axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig("encoding.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 🔤 G

### G — Gradient Descent
**Definition:** An iterative optimisation algorithm that adjusts model parameters to minimise a loss function by moving in the direction of the negative gradient.

$$\theta := \theta - \alpha \cdot \nabla_{\theta} J(\theta)$$

Where α (alpha) is the **learning rate** — the step size.

| Variant | Batch Size | Speed | Stability |
|---------|-----------|-------|-----------|
| **Batch GD** | Full dataset | Slow | Very stable |
| **Stochastic GD (SGD)** | 1 sample | Fast | Noisy |
| **Mini-Batch GD** | 32–512 samples | Balanced | Good (default in deep learning) |


In [ ]:
# Visualise gradient descent on a simple loss surface (MSE)
def loss(theta, X, y):
    return np.mean((X @ theta - y) ** 2)

np.random.seed(42)
X_gd = np.column_stack([np.ones(100), np.random.randn(100)])
true_theta = np.array([2.0, 3.5])
y_gd = X_gd @ true_theta + np.random.randn(100) * 0.5

# Run gradient descent
def gradient_descent(X, y, lr=0.1, n_iter=80):
    theta = np.array([0.0, 0.0])
    history = [theta.copy()]
    for _ in range(n_iter):
        grad  = 2 * X.T @ (X @ theta - y) / len(y)
        theta = theta - lr * grad
        history.append(theta.copy())
    return np.array(history)

# Grid of loss values
t0 = np.linspace(-1, 5, 100)
t1 = np.linspace(0, 6, 100)
T0, T1 = np.meshgrid(t0, t1)
Z = np.array([[loss(np.array([a, b]), X_gd, y_gd) for a in t0] for b in t1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Gradient Descent — Learning Rate Comparison", fontsize=13, fontweight='bold')

for ax, lr, title, col in zip(axes,
        [0.01, 0.25],
        ["LR = 0.01 (Too Slow)", "LR = 0.25 (Good)"],
        [PALETTE[0], PALETTE[2]]):
    hist = gradient_descent(X_gd, y_gd, lr=lr, n_iter=80)
    ax.contourf(T0, T1, Z, levels=30, cmap='Blues', alpha=0.6)
    ax.contour(T0, T1, Z, levels=20, colors='white', alpha=0.3, linewidths=0.5)
    ax.plot(hist[:,0], hist[:,1], 'o-', color=col, linewidth=1.5, markersize=3, label=f'Path (lr={lr})')
    ax.plot(*true_theta, '*', color='gold', markersize=14, zorder=10, label='True minimum')
    ax.set_xlabel("θ₀"); ax.set_ylabel("θ₁")
    ax.set_title(title, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig("gradient_descent.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 🔤 H

### H — Hyperparameter Tuning
**Parameters** are learned during training (e.g. weights).  
**Hyperparameters** are set *before* training (e.g. learning rate, tree depth).

Tuning methods:
- **Grid Search** — exhaustive search over a grid of values
- **Random Search** — samples random combinations (faster, often as good)
- **Bayesian Optimisation** — builds a probabilistic model to guide search


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
X_tr = StandardScaler().fit_transform(X_tr)
X_te_s = StandardScaler().fit(X_tr).transform(X_te)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [3, 5, None],
    'min_samples_split': [2, 5, 10],
}

gs = GridSearchCV(RandomForestClassifier(random_state=42),
                  param_grid, cv=5, scoring='accuracy', n_jobs=-1)
gs.fit(X_tr, y_tr)

print(f"Best parameters : {gs.best_params_}")
print(f"Best CV accuracy: {gs.best_score_:.4f}")
print(f"Test accuracy   : {accuracy_score(y_te, gs.best_estimator_.predict(X_te)):.4f}")

# Visualise one parameter sweep
results_df = pd.DataFrame(gs.cv_results_)
pivot = results_df.pivot_table(values='mean_test_score',
                                index='param_max_depth',
                                columns='param_n_estimators')
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn', ax=ax)
ax.set_title("GridSearchCV — Accuracy Heatmap (max_depth vs n_estimators)", fontweight='bold')
plt.tight_layout()
plt.savefig("grid_search.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 🔤 I

### I — Missing Data & Imputation
Real-world data almost always has missing values. Imputation fills them in rather than dropping rows.

| Strategy | Replace with | Best when |
|----------|-------------|-----------|
| Mean | Feature mean | Numerical, symmetric distribution |
| Median | Feature median | Numerical, skewed or with outliers |
| Most Frequent | Mode | Categorical |
| Constant | A fixed value | Domain-specific |


In [ ]:
# Create a dataset with deliberate missing values
np.random.seed(42)
n = 200
df_miss = pd.DataFrame({
    'Age':    np.random.randint(20, 70, n).astype(float),
    'Income': np.random.normal(50000, 15000, n),
    'Score':  np.random.randint(0, 100, n).astype(float),
})
# Inject missing values (~15%)
for col in df_miss.columns:
    idx = np.random.choice(df_miss.index, size=int(0.15*n), replace=False)
    df_miss.loc[idx, col] = np.nan

print(f"Missing values before imputation:\n{df_miss.isnull().sum()}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Before vs After Mean Imputation", fontsize=13, fontweight='bold')

for ax, col, color in zip(axes, df_miss.columns, PALETTE[:3]):
    imp = SimpleImputer(strategy='mean')
    filled = imp.fit_transform(df_miss[[col]]).ravel()
    ax.hist(df_miss[col].dropna(), bins=20, alpha=0.5, color='gray', label='Original (no NaN)')
    ax.hist(filled, bins=20, alpha=0.5, color=color, label='After imputation')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("imputation.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"\nMissing values after imputation: 0 (all filled)")


---
## 🔤 K

### K1 — K-Nearest Neighbours (KNN)
**Idea:** Classify a new point by looking at the k closest training points and taking a majority vote.  
**Key insight:** Larger k = smoother decision boundary (less variance, more bias). Smaller k = complex boundary (more variance, less bias).


In [ ]:
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                             n_clusters_per_class=1, random_state=42)
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("KNN Decision Boundaries — Effect of k", fontsize=13, fontweight='bold')

xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
mesh = np.c_[xx.ravel(), yy.ravel()]

for ax, k, col in zip(axes, [1, 5, 15, 50], PALETTE):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_s, y)
    Z = knn.predict(mesh).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_s[:,0], X_s[:,1], c=y, cmap='RdBu', edgecolors='white', s=20, linewidth=0.3)
    acc = accuracy_score(y, knn.predict(X_s))
    ax.set_title(f"k = {k}\nTrain Acc: {acc:.2%}", fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig("knn_boundaries.png", dpi=120, bbox_inches='tight')
plt.show()


### K2 — K-Means Clustering
An **unsupervised** algorithm that partitions data into k clusters by minimising the within-cluster sum of squared distances to the cluster centroid.

**Elbow Method:** Plot inertia (total within-cluster variance) vs k. The "elbow" — where adding more clusters stops helping much — is the optimal k.


In [ ]:
X_clust, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=42)

# Elbow method
inertias = []
k_range  = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust)
    inertias.append(km.inertia_)

# Fit best k=4
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km4.fit_predict(X_clust)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(k_range, inertias, 'o-', color=PALETTE[0], linewidth=2)
axes[0].axvline(4, color=PALETTE[1], linestyle='--', label='Optimal k=4')
axes[0].set_title("Elbow Method", fontweight='bold')
axes[0].set_xlabel("Number of Clusters (k)"); axes[0].set_ylabel("Inertia")
axes[0].legend()

for c in range(4):
    mask = labels == c
    axes[1].scatter(X_clust[mask,0], X_clust[mask,1], color=PALETTE[c], alpha=0.6, s=20, label=f'Cluster {c}')
axes[1].scatter(km4.cluster_centers_[:,0], km4.cluster_centers_[:,1],
                c='black', marker='X', s=150, zorder=10, label='Centroids')
axes[1].set_title("K-Means (k=4)", fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("kmeans.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 🔤 L

### L1 — Linear Regression
Models the relationship between a continuous target y and features X as a straight line (or hyperplane in higher dimensions).

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \ldots$$

**Loss function:** Mean Squared Error (MSE)  
$$MSE = \frac{1}{n} \sum (y_i - \hat{y}_i)^2$$


In [ ]:
np.random.seed(42)
X_lin  = np.random.uniform(0, 10, 100).reshape(-1, 1)
y_lin  = 3.5 * X_lin.ravel() + 7 + np.random.normal(0, 4, 100)
X_tr, X_te, y_tr, y_te = train_test_split(X_lin, y_lin, test_size=0.25, random_state=42)

lr_model = LinearRegression()
lr_model.fit(X_tr, y_tr)
y_hat = lr_model.predict(X_te)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(X_tr, y_tr, alpha=0.5, color=PALETTE[0], label='Train', s=25)
axes[0].scatter(X_te, y_te, alpha=0.5, color=PALETTE[1], label='Test', s=25)
x_line = np.linspace(0, 10, 100).reshape(-1,1)
axes[0].plot(x_line, lr_model.predict(x_line), 'k-', linewidth=2, label='Regression line')
axes[0].set_title("Linear Regression", fontweight='bold')
axes[0].set_xlabel("X"); axes[0].set_ylabel("y")
axes[0].legend()

# Residual plot
residuals = y_te - y_hat
axes[1].scatter(y_hat, residuals, alpha=0.6, color=PALETTE[2], s=25)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title("Residual Plot", fontweight='bold')
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residuals")

plt.tight_layout()
plt.savefig("linear_regression.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"Coefficients: slope={lr_model.coef_[0]:.3f}, intercept={lr_model.intercept_:.3f}")
print(f"MSE: {mean_squared_error(y_te, y_hat):.3f}  |  R²: {r2_score(y_te, y_hat):.4f}")


---
## 🔤 O

### O — Overfitting & Underfitting

| State | Description | Symptoms | Fix |
|-------|-------------|---------|-----|
| **Underfitting** | Model is too simple | High train error, high test error | More features, more complex model, less regularisation |
| **Good fit** | Model generalises well | Low train error, low test error | — |
| **Overfitting** | Model memorises training data | Very low train error, high test error | More data, regularisation, dropout, simpler model |


In [ ]:
np.random.seed(42)
X_of = np.sort(np.random.uniform(0, 1, 40))
y_of = np.sin(2 * np.pi * X_of) + np.random.normal(0, 0.15, 40)
X_of_col = X_of.reshape(-1, 1)

train_errs, test_errs = [], []
degrees = list(range(1, 20))
X_tr_of, X_te_of, y_tr_of, y_te_of = train_test_split(X_of_col, y_of, test_size=0.35, random_state=42)

for deg in degrees:
    pipe = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    pipe.fit(X_tr_of, y_tr_of)
    train_errs.append(mean_squared_error(y_tr_of, pipe.predict(X_tr_of)))
    test_errs.append(mean_squared_error(y_te_of, pipe.predict(X_te_of)))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Overfitting vs Underfitting", fontsize=14, fontweight='bold')

x_smooth = np.linspace(0, 1, 300).reshape(-1, 1)
for ax, (deg, label, col) in zip(axes, [
        (1,  "Underfitting
(degree=1)",   PALETTE[0]),
        (4,  "Good Fit
(degree=4)",        PALETTE[2]),
        (18, "Overfitting
(degree=18)",    PALETTE[1])]):
    pipe = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    pipe.fit(X_tr_of, y_tr_of)
    ax.scatter(X_tr_of, y_tr_of, color='gray', alpha=0.7, s=30, zorder=5)
    ax.plot(x_smooth, np.sin(2*np.pi*x_smooth), 'k--', linewidth=1.5, label='True')
    ax.plot(x_smooth, pipe.predict(x_smooth), color=col, linewidth=2.5, label=f'Degree {deg}')
    tr_e = mean_squared_error(y_tr_of, pipe.predict(X_tr_of))
    te_e = mean_squared_error(y_te_of, pipe.predict(X_te_of))
    ax.set_title(f"{label}\nTrain MSE: {tr_e:.3f} | Test MSE: {te_e:.3f}", fontsize=10, fontweight='bold')
    ax.set_ylim(-2.5, 2.5); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("overfit_underfit.png", dpi=120, bbox_inches='tight')
plt.show()

# Learning curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(degrees, train_errs, 'o-', color=PALETTE[0], linewidth=2, label='Train MSE')
ax.semilogy(degrees, test_errs,  'o-', color=PALETTE[1], linewidth=2, label='Test MSE')
ax.axvline(4, color=PALETTE[2], linestyle='--', label='Sweet spot (degree=4)')
ax.set_xlabel("Polynomial Degree (Model Complexity)")
ax.set_ylabel("MSE (log scale)")
ax.set_title("Bias–Variance in Action: Train vs Test Error", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig("overfit_curve.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 🔤 P

### P1 — Pipelines
A `Pipeline` chains preprocessing steps and a model into a single object. This prevents **data leakage** (fitting the scaler on test data) and makes code cleaner.

**Data Leakage:** Fitting a scaler on the whole dataset before splitting means the scaler "sees" test data during training — inflating performance estimates.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

# ── WRONG: Leaky approach ─────────────────────────────────
scaler_leaky = StandardScaler()
X_all_scaled = scaler_leaky.fit_transform(X)          # sees ALL data
X_tr_leak, X_te_leak, y_tr_l, y_te_l = train_test_split(X_all_scaled, y, test_size=0.25, random_state=42)
clf_leak = RandomForestClassifier(n_estimators=100, random_state=42)
clf_leak.fit(X_tr_leak, y_tr_l)
leaky_acc = accuracy_score(y_te_l, clf_leak.predict(X_te_leak))

# ── CORRECT: Pipeline ─────────────────────────────────────
pipe_correct = Pipeline([
    ('scaler',     StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipe_correct.fit(X_tr, y_tr)                          # scaler only sees X_tr
correct_acc = accuracy_score(y_te, pipe_correct.predict(X_te))

print(f"Leaky accuracy  : {leaky_acc:.4f}  ← Inflated!")
print(f"Correct accuracy: {correct_acc:.4f}  ← Trustworthy")

# Show pipeline structure
print("\nPipeline steps:")
for name, step in pipe_correct.steps:
    print(f"  [{name}] → {step.__class__.__name__}")


---
## 🔤 R

### R1 — Regularisation (L1 Lasso & L2 Ridge)
Regularisation adds a penalty to the loss function to discourage large coefficients, reducing overfitting.

| | L1 (Lasso) | L2 (Ridge) |
|--|-----------|-----------|
| **Penalty** | Σ\|θᵢ\| | Σθᵢ² |
| **Effect** | Drives coefficients to exactly 0 → feature selection | Shrinks coefficients toward 0, never exactly 0 |
| **Best for** | Sparse models, many irrelevant features | When all features contribute some signal |


In [ ]:
from sklearn.datasets import make_regression
X_reg, y_reg, coef_true = make_regression(n_samples=100, n_features=20, n_informative=5,
                                           noise=20, coef=True, random_state=42)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.25, random_state=42)

models_reg = {
    'Linear (no reg)': LinearRegression(),
    'Ridge (L2, α=1)': Ridge(alpha=1.0),
    'Ridge (L2, α=100)': Ridge(alpha=100.0),
    'Lasso (L1, α=1)': Lasso(alpha=1.0),
    'Lasso (L1, α=10)': Lasso(alpha=10.0),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_cycle = PALETTE * 3

coef_data, names_data = [], []
for name, m in models_reg.items():
    m.fit(X_tr_r, y_tr_r)
    coef_data.append(m.coef_)
    names_data.append(name)

x_pos = np.arange(20)
for i, (coefs, label) in enumerate(zip(coef_data, names_data)):
    axes[0].plot(x_pos, np.abs(coefs), 'o-', alpha=0.7, label=label, linewidth=1.5, markersize=4)
axes[0].set_title("Coefficient Magnitude by Model", fontweight='bold')
axes[0].set_xlabel("Feature index"); axes[0].set_ylabel("|Coefficient|")
axes[0].legend(fontsize=7)

# Sparsity — count zeros (threshold < 1e-4)
n_zeros = [(name, np.sum(np.abs(m.coef_) < 1e-4)) for name, m in models_reg.items()]
names_z, zeros_z = zip(*n_zeros)
axes[1].barh(names_z, zeros_z, color=PALETTE[:len(names_z)], edgecolor='white')
axes[1].set_title("Number of Near-Zero Coefficients (Sparsity)", fontweight='bold')
axes[1].set_xlabel("Count of ~zero coefficients")
plt.tight_layout()
plt.savefig("regularisation.png", dpi=120, bbox_inches='tight')
plt.show()
print("💡 Lasso pushes coefficients to exactly zero — effectively removing features.")


### R2 — ROC Curve & AUC
The **ROC (Receiver Operating Characteristic)** curve plots True Positive Rate vs False Positive Rate at every classification threshold.  
**AUC (Area Under the Curve):** 1.0 = perfect, 0.5 = random guessing.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.50)')

models_roc = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5),
    "SVM":                 SVC(probability=True, kernel='rbf'),
}
scaler_roc = StandardScaler()
X_tr_s = scaler_roc.fit_transform(X_tr)
X_te_s = scaler_roc.transform(X_te)

for (name, m), col in zip(models_roc.items(), PALETTE):
    m.fit(X_tr_s, y_tr)
    proba = m.predict_proba(X_te_s)[:,1]
    fpr, tpr, _ = roc_curve(y_te, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2, color=col, label=f"{name} (AUC={roc_auc:.3f})")

ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Breast Cancer Dataset", fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 🔤 S

### S1 — Support Vector Machines (SVM)
SVM finds the **maximum-margin hyperplane** that best separates classes.  
Support vectors are the data points closest to the decision boundary — they define it.

**Kernel trick:** Projects data into higher dimensions to find a linear separator for non-linearly separable data.

| Kernel | Best for |
|--------|---------|
| Linear | Linearly separable data, high-dimensional text |
| RBF (Gaussian) | Non-linear, general purpose |
| Polynomial | Image classification |


In [ ]:
# Generate separable data
X_svm, y_svm = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, random_state=42)
X_svm_s = StandardScaler().fit_transform(X_svm)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("SVM — Effect of Kernel Choice", fontsize=13, fontweight='bold')

xx_s, yy_s = np.meshgrid(np.linspace(-3,3,200), np.linspace(-3,3,200))
mesh_s = np.c_[xx_s.ravel(), yy_s.ravel()]

for ax, (kernel, C), col in zip(axes,
        [('linear', 1), ('rbf', 1), ('rbf', 100)],
        PALETTE[:3]):
    title = f"Kernel={kernel}, C={C}"
    svm = SVC(kernel=kernel, C=C, probability=True)
    svm.fit(X_svm_s, y_svm)
    Z_s = svm.predict(mesh_s).reshape(xx_s.shape)
    ax.contourf(xx_s, yy_s, Z_s, alpha=0.3, cmap='RdBu')
    ax.scatter(X_svm_s[:,0], X_svm_s[:,1], c=y_svm, cmap='RdBu',
               edgecolors='white', s=20, linewidth=0.3)
    # Mark support vectors
    sv = svm.support_vectors_
    ax.scatter(sv[:,0], sv[:,1], s=80, facecolors='none',
               edgecolors='black', linewidths=1.5, label='Support vectors')
    acc = accuracy_score(y_svm, svm.predict(X_svm_s))
    ax.set_title(f"{title}\nAcc: {acc:.2%}", fontsize=10, fontweight='bold')
    ax.legend(fontsize=8); ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig("svm.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 🔤 T

### T — Train / Validation / Test Split

| Split | Purpose | Typical Size |
|-------|---------|-------------|
| **Training set** | Fit model parameters | 60–70 % |
| **Validation set** | Tune hyperparameters | 10–15 % |
| **Test set** | Final unbiased evaluation — touch it ONCE | 20–25 % |

⚠️ **Golden Rule:** Never use the test set to make modelling decisions. It must only be used for the final evaluation report.


In [ ]:
# Visualise the split proportions
X, y = make_classification(n_samples=1000, random_state=42)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1875, random_state=42)
# 0.1875 of 0.80 = 0.15 of total → 60/15/25 split

print(f"Training samples  : {len(X_train):4d}  ({len(X_train)/len(X):.0%})")
print(f"Validation samples: {len(X_val):4d}  ({len(X_val)/len(X):.0%})")
print(f"Test samples      : {len(X_test):4d}  ({len(X_test)/len(X):.0%})")

# Train, tune, evaluate cycle
pipeline = Pipeline([('scaler', StandardScaler()),
                     ('model',  RandomForestClassifier(random_state=42))])

best_acc, best_n = 0, 0
for n_est in [10, 50, 100, 200]:
    pipeline.set_params(model__n_estimators=n_est)
    pipeline.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, pipeline.predict(X_val))
    if val_acc > best_acc:
        best_acc, best_n = val_acc, n_est
    print(f"  n_estimators={n_est:>3}  val_acc={val_acc:.4f}")

# Final evaluation on untouched test set
pipeline.set_params(model__n_estimators=best_n)
pipeline.fit(X_train, y_train)
test_acc = accuracy_score(y_test, pipeline.predict(X_test))
print(f"\nBest n_estimators: {best_n} (val acc: {best_acc:.4f})")
print(f"FINAL Test Accuracy: {test_acc:.4f}  ← This is the number you report")


---
## 🔤 V

### V — Validation Curves
A validation curve shows how training and cross-validation scores change as a single hyperparameter varies. It helps diagnose overfitting vs underfitting for that parameter.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_s = StandardScaler().fit_transform(X)

param_range = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30]
train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=42), X_s, y,
    param_name='max_depth', param_range=param_range,
    cv=5, scoring='accuracy', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(param_range, train_mean, 'o-', color=PALETTE[0], linewidth=2, label='Training score')
ax.fill_between(param_range, train_mean - train_std, train_mean + train_std, alpha=0.15, color=PALETTE[0])
ax.plot(param_range, val_mean, 'o-', color=PALETTE[1], linewidth=2, label='Cross-validation score')
ax.fill_between(param_range, val_mean - val_std, val_mean + val_std, alpha=0.15, color=PALETTE[1])

best_depth = param_range[np.argmax(val_mean)]
ax.axvline(best_depth, color=PALETTE[2], linestyle='--', label=f'Best depth={best_depth}')
ax.set_title("Validation Curve — Decision Tree max_depth", fontweight='bold')
ax.set_xlabel("max_depth"); ax.set_ylabel("Accuracy")
ax.legend(); ax.set_ylim(0.88, 1.02)

# Annotate regions
ax.annotate("Underfitting", xy=(1, 0.91), fontsize=10, color=PALETTE[0])
ax.annotate("Overfitting →", xy=(15, 0.93), fontsize=10, color=PALETTE[1])

plt.tight_layout()
plt.savefig("validation_curve.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 🔤 W

### W — Feature Importance & Model Weights
Understanding which features drive predictions is critical for model interpretation, debugging, and stakeholder communication.

- **Linear models:** coefficients represent the change in output per unit change in that feature (after scaling)
- **Tree-based models:** feature importance = total reduction in Gini impurity attributed to splits on that feature


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
feature_names = load_breast_cancer().feature_names
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr_s, y_tr)
lr = LogisticRegression(max_iter=500, C=0.5)
lr.fit(X_tr_s, y_tr)

# Top 10 features from each
top_n = 10
rf_idx = np.argsort(rf.feature_importances_)[::-1][:top_n]
lr_idx = np.argsort(np.abs(lr.coef_[0]))[::-1][:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(range(top_n), rf.feature_importances_[rf_idx][::-1], color=PALETTE[0], edgecolor='white')
axes[0].set_yticks(range(top_n)); axes[0].set_yticklabels([feature_names[i] for i in rf_idx[::-1]], fontsize=8)
axes[0].set_title("Random Forest — Feature Importances", fontweight='bold')
axes[0].set_xlabel("Importance")

axes[1].barh(range(top_n), lr.coef_[0][lr_idx][::-1], color=PALETTE[1], edgecolor='white')
axes[1].set_yticks(range(top_n)); axes[1].set_yticklabels([feature_names[i] for i in lr_idx[::-1]], fontsize=8)
axes[1].set_title("Logistic Regression — Coefficient Magnitude", fontweight='bold')
axes[1].set_xlabel("|Coefficient|")
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 💾 Model Persistence — Saving & Loading Models
After training, save your model with `joblib` so you don't have to retrain it every time.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

final_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(n_estimators=100, random_state=42))
])
final_pipe.fit(X_tr, y_tr)

# Save
joblib.dump(final_pipe, '/tmp/paybridge_model.pkl')
print("Model saved to /tmp/paybridge_model.pkl")

# Load and verify
loaded_model = joblib.load('/tmp/paybridge_model.pkl')
acc = accuracy_score(y_te, loaded_model.predict(X_te))
print(f"Loaded model test accuracy: {acc:.4f}")
print("\n✅ The loaded model produces identical predictions to the original.")


---
## 📋 Quick Reference Cheat Sheet

### When to Use Which Algorithm

| Problem Type | Small Dataset | Large Dataset |
|-------------|--------------|--------------|
| Binary Classification | Logistic Regression, SVM | Random Forest, Gradient Boosting |
| Multi-class Classification | Decision Tree, KNN | Random Forest, Neural Networks |
| Regression | Linear/Ridge/Lasso | Gradient Boosting, Neural Networks |
| Clustering (no labels) | K-Means, DBSCAN | Mini-Batch K-Means |
| Dimensionality Reduction | PCA, t-SNE | PCA, UMAP |

### Scaling Decision Table

| Algorithm | Needs Scaling? | Preferred Scaler |
|-----------|---------------|-----------------|
| KNN | ✅ Yes | StandardScaler or MinMaxScaler |
| SVM | ✅ Yes | StandardScaler |
| Neural Networks | ✅ Yes | MinMaxScaler [0,1] |
| Logistic Regression | ✅ Yes (with regularisation) | StandardScaler |
| Decision Trees | ❌ No | — |
| Random Forest | ❌ No | — |
| Gradient Boosting | ❌ No | — |

### Encoding Decision Table

| Category type | Example | Use |
|---------------|---------|-----|
| Binary | Yes/No | Label Encoding |
| Ordinal | Small < Medium < Large | Label / Ordinal Encoding |
| Nominal (low cardinality) | City (< 10 cities) | One-Hot Encoding |
| Nominal (high cardinality) | Zip codes (1000+) | Target Encoding / Embeddings |

---

> *"All models are wrong, but some are useful."* — George Box  
> The goal is not a perfect model — it's one that's good enough to make reliable, interpretable decisions on new data.
